In [ ]:
# ============================================================
# CELL 1 — CONFIG 
# ============================================================

# History: round 1 = 4,000 clips (base -> sft) | round 2 = 25,000 (-> v2)
#          round 3 = 34,000 (-> v3, continued from v2)
#          round 4 = 34,000, FRESH from base, with the same speaker concat fix

# --- Dataset sampling ---
N_SAMPLES = 34000          
MIN_UP_VOTES = 2           # quality filter
MAX_DOWN_VOTES = 0
MIN_SENTENCE_LEN = 10
MAX_SENTENCE_LEN = 200

# --- Multi sentence concatenation ---
# Groups are now built from a single speaker's clips only.
# Lower this to 0.2 if CELL 6 reports far fewer groups than requested.
CONCAT_FRACTION = 0.3
CONCAT_MIN_CLIPS = 2
CONCAT_MAX_CLIPS = 3

# --- Training ---
# Fresh start from the base model. Three rounds of stacked epochs on
# overlapping data was giving diminishing returns.
BASE_MODEL_PATH = "./models/MOSS-TTS-Nano"

CODEC_PATH = "./models/MOSS-Audio-Tokenizer-Nano"
OUTPUT_DIR = "./output/moss_tts_nano_sft_v4"   # new folder, v1/v2/v3 stay intact
NUM_EPOCHS = 3
LEARNING_RATE = 1e-5
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
MAX_LENGTH = 1024
NUM_PROCESSES = 2           # both T4s, safe in fp32 with no grad scaler conflict
MIXED_PRECISION = "no"      # T4 has no bf16, fp16 broke grad scaling, fp32 works

# --- Paths (shouldn't need to touch these) ---
KAGGLE_DATASET = "amirftma/common-voice-fa-v13"
WORKDIR = "MOSS-TTS-Nano"

In [ ]:
# ============================================================
# CELL 2 — Imports & installs 
# ============================================================
import os
import sys
import json
import random
import subprocess

import pandas as pd
import IPython.display as ipd

def run(cmd, cwd=None):
    """Stream subprocess output live instead of buffering it silently."""
    print(f"$ {cmd}")
    return os.system(f"cd {cwd} && {cmd}" if cwd else cmd)

# Core deps
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "accelerate", "datasets", "soundfile", "kagglehub"], check=True)

# Clone the repo (skip if already present from a prior session)
if not os.path.isdir(WORKDIR):
    subprocess.run(["git", "clone", "https://github.com/OpenMOSS/MOSS-TTS-Nano.git", WORKDIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{WORKDIR}/requirements.txt"], check=True)
    # Fix the torch/torchvision version mismatch discovered earlier
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchvision==0.22.0"], check=True)

import torch
print("CUDA available:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())

In [ ]:
# ============================================================
# CELL 3 — Download base model + codec (skips if already downloaded)
# ============================================================
from huggingface_hub import snapshot_download

local_tts_path = os.path.join(WORKDIR, "models", "MOSS-TTS-Nano")
local_codec_path = os.path.join(WORKDIR, "models", "MOSS-Audio-Tokenizer-Nano")

if not os.path.exists(local_tts_path):
    snapshot_download(repo_id="OpenMOSS-Team/MOSS-TTS-Nano", local_dir=local_tts_path)
if not os.path.exists(local_codec_path):
    snapshot_download(repo_id="OpenMOSS-Team/MOSS-Audio-Tokenizer-Nano", local_dir=local_codec_path)

print("Model:", local_tts_path)
print("Codec:", local_codec_path)

In [ ]:
# ============================================================
# CELL 4 — Download & load the Common Voice dataset
# ============================================================
import kagglehub

dataset_path = kagglehub.dataset_download(KAGGLE_DATASET)
corpus_root = os.path.join(dataset_path, "corpus")

df = pd.read_csv(os.path.join(corpus_root, "validated.tsv"), sep="\t")
df["sentence_len"] = df["sentence"].str.len()

df_clean = df[
    (df["up_votes"] >= MIN_UP_VOTES) &
    (df["down_votes"] == MAX_DOWN_VOTES) &
    (df["sentence_len"] >= MIN_SENTENCE_LEN) &
    (df["sentence_len"] <= MAX_SENTENCE_LEN)
].copy()

print(f"Quality pool: {len(df_clean)} clips (out of {len(df)} total)")
# Output: Quality pool: 281084 clips (out of 304410 total)

In [ ]:
# ============================================================
# CELL 5 — Build the training set: single-sentence + concatenated multi-sentence
# ============================================================
from tqdm.auto import tqdm

with tqdm(total=3, desc="Building training set split") as pbar:
    n_concat_groups = int((N_SAMPLES * CONCAT_FRACTION) / ((CONCAT_MIN_CLIPS + CONCAT_MAX_CLIPS) / 2))
    n_single = N_SAMPLES - int(N_SAMPLES * CONCAT_FRACTION)
    pbar.update(1)

    pool = df_clean.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle once
    pbar.update(1)

    single_df = pool.iloc[:n_single]
    concat_pool = pool.iloc[n_single:]
    pbar.update(1)

print(f"Single-sentence examples: {len(single_df)}")
print(f"Concatenated-group examples: ~{n_concat_groups} (from {len(concat_pool)} spare clips)")
# Output: Single-sentence examples: 17500
# Concatenated-group examples: ~3000 (from 263584 spare clips)

In [ ]:
# ============================================================
# CELL 6 — Convert clips to wav, build both example types, write JSONL
# ============================================================
import glob
from tqdm.auto import tqdm

wav_dir = "/kaggle/working/train_wavs"
os.makedirs(wav_dir, exist_ok=True)

# Delete concat files from earlier rounds. Those mixed several speakers into
# one clip and must not survive into this run.
stale = glob.glob(os.path.join(wav_dir, "concat_*.wav"))
for p in stale:
    os.remove(p)
print(f"Removed {len(stale)} stale concat files from previous runs")

def to_wav(mp3_path, wav_path):
    if not os.path.exists(wav_path):
        subprocess.run(["ffmpeg", "-y", "-i", mp3_path, "-ar", "24000", "-ac", "1", wav_path],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return os.path.exists(wav_path)

records = []

# --- single sentence examples ---
for _, row in tqdm(single_df.iterrows(), total=len(single_df), desc="Converting single sentence clips"):
    mp3_path = os.path.join(corpus_root, "clips", row["path"])
    wav_path = os.path.join(wav_dir, row["path"].replace(".mp3", ".wav"))
    if to_wav(mp3_path, wav_path):
        records.append({"audio": wav_path, "text": row["sentence"], "language": "fa"})

print(f"Single sentence converted: {len(records)}")

# --- concatenated examples, ONE SPEAKER PER GROUP ---
# The old version sliced a globally shuffled pool, so nearly every group
# spliced 2 or 3 different people into one utterance. That taught the model
# the voice may change mid clip, which is exactly what we do not want.
concat_records = []
concat_pool = concat_pool.reset_index(drop=True)

speaker_groups = []
for cid, g in concat_pool.groupby("client_id"):
    rows = g.to_dict("records")
    random.shuffle(rows)
    while len(rows) >= CONCAT_MIN_CLIPS:
        n = random.randint(CONCAT_MIN_CLIPS, min(CONCAT_MAX_CLIPS, len(rows)))
        speaker_groups.append([rows.pop() for _ in range(n)])

random.shuffle(speaker_groups)
print(f"Same speaker groups available: {len(speaker_groups)} (requested {n_concat_groups})")
speaker_groups = speaker_groups[:n_concat_groups]

group_id = 0
for group in tqdm(speaker_groups, desc="Building same speaker concat groups"):
    wav_paths, texts, ok = [], [], True
    for row in group:
        mp3_path = os.path.join(corpus_root, "clips", row["path"])
        wav_path = os.path.join(wav_dir, row["path"].replace(".mp3", ".wav"))
        if not to_wav(mp3_path, wav_path):
            ok = False
            break
        wav_paths.append(wav_path)
        texts.append(row["sentence"])
    if not ok:
        continue

    combined_wav = os.path.join(wav_dir, f"concat_{group_id:05d}.wav")
    inputs = "".join(f'-i "{p}" ' for p in wav_paths)
    filter_parts = "".join(f"[{i}:a]" for i in range(len(wav_paths)))
    concat_cmd = (
        f'ffmpeg -y {inputs} -filter_complex "{filter_parts}concat=n={len(wav_paths)}:v=0:a=1[out]" '
        f'-map "[out]" "{combined_wav}"'
    )
    subprocess.run(concat_cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if os.path.exists(combined_wav):
        concat_records.append({"audio": combined_wav, "text": " ".join(texts), "language": "fa"})
        group_id += 1

print(f"Concatenated groups built: {len(concat_records)}")

all_records = records + concat_records
random.shuffle(all_records)

out_path = "/kaggle/working/train_raw.jsonl"
with open(out_path, "w", encoding="utf-8") as f:
    for r in all_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Total training examples written: {len(all_records)} -> {out_path}")

In [ ]:
# ============================================================
# CELL 7 — Sanity-check a few examples by ear before committing to prep+training
# ============================================================
for r in random.sample(all_records, 3):
    print(r["text"])
    display(ipd.Audio(r["audio"]))

In [ ]:
# ============================================================
# CELL 8 — Preprocess: encode audio into audio_codes
# ============================================================
import shutil
import subprocess
import threading
import time
from tqdm.auto import tqdm

shutil.copy("/kaggle/working/train_raw.jsonl", os.path.join(WORKDIR, "train_raw.jsonl"))

def run_with_live_progress(cmd, cwd):
    """
    Streams the subprocess's own output live (instead of buffering until exit),
    and shows an elapsed-time bar alongside it. Note: this is a time elapsed
    indicator, not a true percent-complete bar — prepare_data.py doesn't expose
    per-batch progress, so we can't calculate a real percentage without editing
    that script directly.
    """
    process = subprocess.Popen(
        cmd, shell=True, cwd=cwd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )

    stop_flag = {"done": False}

    def tick_timer():
        with tqdm(desc="Encoding audio_codes (elapsed)", bar_format="{desc}: {elapsed}") as pbar:
            while not stop_flag["done"]:
                pbar.refresh()
                time.sleep(1)

    timer_thread = threading.Thread(target=tick_timer, daemon=True)
    timer_thread.start()

    for line in process.stdout:
        print(line, end="")

    process.wait()
    stop_flag["done"] = True
    timer_thread.join()
    return process.returncode

returncode = run_with_live_progress(
    f"python finetuning/prepare_data.py "
    f"--codec-path {CODEC_PATH} "
    f"--input-jsonl train_raw.jsonl "
    f"--output-jsonl train_with_codes.jsonl "
    f"--batch-size 8",
    cwd=WORKDIR
)
print(f"\nExit code: {returncode}")

In [ ]:
# ============================================================
# CELL 9 — Train
# ============================================================
run(
    f"accelerate launch --num_processes {NUM_PROCESSES} --mixed_precision {MIXED_PRECISION} "
    f"finetuning/sft.py "
    f"--model-path {BASE_MODEL_PATH} "
    f"--codec-path {CODEC_PATH} "
    f"--train-jsonl train_with_codes.jsonl "
    f"--output-dir {OUTPUT_DIR} "
    f"--per-device-batch-size {PER_DEVICE_BATCH_SIZE} "
    f"--gradient-accumulation-steps {GRAD_ACCUM_STEPS} "
    f"--learning-rate {LEARNING_RATE} "
    f"--warmup-ratio 0.03 "
    f"--num-epochs {NUM_EPOCHS} "
    f"--mixed-precision {MIXED_PRECISION} "
    f"--max-length {MAX_LENGTH} "
    f"--channelwise-loss-weight 1,32",
    cwd=WORKDIR
)